# Week 6 checkpoint and velocity-metric audit

This notebook performs CPU inference only. It does not train, use OOD data, or change the original `0.01` median velocity-nRMSE gate. Attach the private dataset containing `chronopde.h5` and the saved mechanics output containing the `.pt` checkpoints. Internet must be enabled for the pinned repository checkout; no accelerator is required.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import traceback
import uuid
from pathlib import Path

from IPython.display import FileLink, display

IMPLEMENTATION_COMMIT = 'c7bda5977afe3ad24b204ddc94dda34d3abe87f1'
REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = (
    '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
)
RUN_ID = uuid.uuid4().hex[:8]
REPOSITORY = Path(f'/kaggle/working/Chrono_pde_velocity_audit_{RUN_ID}')
STATE_ROOT = Path(f'/kaggle/working/chronopde_velocity_audit_{RUN_ID}')
REPORT = STATE_ROOT / 'report'
PACKAGE_BASE = Path('/kaggle/working/chronopde_week6_velocity_audit')


In [ ]:
failure = None
return_code = None
STATE_ROOT.mkdir(parents=True, exist_ok=False)
try:
    candidates = [
        path
        for path in Path('/kaggle/input').rglob('chronopde.h5')
        if path.is_file() and path.stat().st_size == EXPECTED_DATA_SIZE
    ]
    if not candidates:
        raise FileNotFoundError('Attach the private dataset containing chronopde.h5')
    DATA = candidates[0]
    digest = hashlib.sha256()
    with DATA.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != EXPECTED_DATA_SHA256:
        raise ValueError('chronopde.h5 checksum does not match the frozen dataset')

    protocol_files = list(Path('/kaggle/input').rglob('diagnostic_protocol.json'))
    checkpoint_directories = [
        path.parent
        for path in protocol_files
        if (path.parent / 'summary.json').is_file()
        and (path.parent / 'metrics.jsonl').is_file()
        and (path.parent / 'best.pt').is_file()
        and (path.parent / 'last.pt').is_file()
    ]
    if len(checkpoint_directories) < 6:
        raise FileNotFoundError(
            'Attach the saved mechanics output containing all six checkpoint directories'
        )
    mechanics_root = Path('/kaggle/input')
    print('Verified dataset:', DATA)
    print('Mechanics runs found:', len(checkpoint_directories))

    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
    subprocess.run(
        ['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT],
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--ignore-requires-python', '-e', '.'],
        cwd=REPOSITORY,
        check=True,
    )
    command = [
        sys.executable,
        'scripts/audit_velocity.py',
        '--config',
        'configs/project.yaml',
        '--data-path',
        str(DATA),
        '--mechanics-root',
        str(mechanics_root),
        '--output-directory',
        str(REPORT),
        '--device',
        'cpu',
    ]
    completed = subprocess.run(command, cwd=REPOSITORY, check=False)
    return_code = completed.returncode
    if return_code != 0:
        raise RuntimeError(f'audit command returned {return_code}')
except Exception as error:
    failure = {
        'error_type': type(error).__name__,
        'message': str(error),
        'return_code': return_code,
        'traceback': traceback.format_exc(),
    }
    (STATE_ROOT / 'failure.json').write_text(
        json.dumps(failure, indent=2) + '\n', encoding='utf-8'
    )
finally:
    package = shutil.make_archive(str(PACKAGE_BASE), 'zip', root_dir=STATE_ROOT)
    print('Audit status:', 'failed' if failure else 'completed')
    print('Package:', package)
    display(FileLink(package))


In [ ]:
summary_path = REPORT / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    print('Original Week 6 median gate changed:', False)
else:
    print((STATE_ROOT / 'failure.json').read_text())
